# Generate heterostructures

Builds coherent film/substrate interfaces directly from **two already-cut slab POSCARs**
(e.g. the ones `generate-slabs.ipynb` produces) -- no bulk unit cell needed. For a given
substrate slab and film slab:

1. `ZSLGenerator` is run directly on the two slabs' in-plane lattice vectors to find
   coincidence (coherent) matches. This is purely linear algebra on the two 2-D lattices --
   it never looks at crystal symmetry, so it treats a cubic/hexagonal pair exactly like an
   orthorhombic/trigonal pair. Lower-symmetry or more mismatched pairs just tend to need a
   larger `max_interface_area` (bigger supercells) or looser tolerances before any match is
   found at all -- see `angle_scan_points`/the printed match counts below to check this for
   your specific pair.
2. The film is optionally pre-rotated about the interface normal before step 1, which is
   how we scan **twist angle**: rotating the film changes which supercell combinations of
   the two lattices happen to coincide, so different twist angles generally expose a
   different, mostly non-overlapping set of matches (this is the same effect used to build
   moiré/twisted-bilayer structures).
3. For each (angle, match), symmetry-distinct **in-plane registries** (lateral stacking
   offsets) are enumerated with `Interface.get_shifts_based_on_adsorbate_sites`, and a small
   grid of **gaps** is applied.

All of this produces a *controlled, finite* set of inequivalent candidate interface
`POSCAR` files -- exactly like `generate-slabs.ipynb` enumerates inequivalent slabs -- with
one parameter per axis (`n_twist_angles`, `n_matches_per_angle`, `n_registries_per_match`,
`n_gaps`) and a single overall cap (`max_total_candidates`) if their product is too large.

Relaxing/ranking those candidates with the MACE potential is a **separate, optional** step
(`run_mliap_relaxation`), gated so it's off by default: every candidate `POSCAR` written by
the generation step is left completely untouched (relaxation results go into a separate
`CONTCAR` next to it, the existing convention in this repo), so the generated set can be
handed straight to VASP/DFT instead if you'd rather not use the MLIAP at all.

**On substrate vs. film:** you do have to designate one slab as substrate and one as film --
it determines the geometry (substrate stays put, film is flipped and stacked on top with
the given gap/vacuum) -- but the match-finding itself (`bidirectional=True` below) searches
supercells starting from either lattice, so calling one "substrate" doesn't bias which
matches are found, only how the final structure is laid out.

**On checking the surface:** a Miller index is only meaningful relative to a bulk cell, so
there's no way to verify from a bare slab `POSCAR` alone that it truly *is* the surface its
filename claims. What we can check is that a file at least *looks* like a slab (a large
vacuum gap along one lattice vector) rather than, say, an accidentally-supplied bulk cell --
see `warn_if_not_slab_like` below. If you also have the bulk structure handy, there's an
optional verification cell near the end that re-cuts a slab from it at the stated Miller
index and compares in-plane lattice parameters against your supplied `POSCAR`.


In [ ]:
import copy
import itertools
import json
import os

import numpy as np
import matplotlib.pyplot as plt

from pymatgen.core.structure import Structure
from pymatgen.core.surface import Slab, SlabGenerator
from pymatgen.core.interface import Interface
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import get_2d_transform, from_2d_to_3d
from pymatgen.analysis.elasticity.strain import Deformation
from pymatgen.transformations.standard_transformations import RotationTransformation

from ase.io.vasp import write_vasp

import libraries.utilities as sul
import libraries.model      as slm


## Inputs -- two already-cut slabs, plus the Miller index each was cut along

In [ ]:
general_folder = 'input/CeO2-heterostructure'

substrate_miller = (3, 1, 1)
substrate_POSCAR = "/Users/cibran/work/UPC/SlabOptimization/input/CONTCAR-0_1_2_i_76"

film_miller = (4, 4, 3)
film_POSCAR = "/Users/cibran/work/UPC/SlabOptimization/input/CONTCAR-0_1_1_i_14"


def warn_if_not_slab_like(structure, label):
    """A bulk unit cell typically has a,b,c of similar magnitude; a slab (with vacuum)
    has a c-axis much longer than a/b. This only catches the "gave it a bulk cell by
    mistake" case -- it can't verify the Miller index itself (that's only meaningful
    relative to a bulk cell, which we don't have here). See the optional verification
    cell near the end of this notebook if you do have the bulk structure available."""
    a, b, c = structure.lattice.abc
    if c < 1.5 * max(a, b):
        print(f"[warning] '{label}' has c={c:.1f} Å vs a/b={a:.1f}/{b:.1f} Å -- this does "
              f"NOT look like a slab with vacuum. Double check {label}_POSCAR is really an "
              f"already-cut slab, not a bulk unit cell.")


def wrap_as_slab(structure, miller_index):
    """Wrap a plain Structure (as read back from a POSCAR) into a Slab object.
    `oriented_unit_cell` and `shift` aren't recoverable from a bare slab file, so they're
    just placeholders -- fine here, since we only use `.lattice`/`.species`/`.frac_coords`
    and `.make_supercell()`/`.get_orthogonal_c_slab()` below, never the bulk-provenance
    metadata. `reorient_lattice=False` so pymatgen doesn't silently re-orient the lattice
    you gave it."""
    return Slab(structure.lattice, structure.species, structure.frac_coords,
                miller_index=miller_index, oriented_unit_cell=structure,
                shift=0.0, scale_factor=np.eye(3), reorient_lattice=False,
                coords_are_cartesian=False)


substrate_structure = Structure.from_file(substrate_POSCAR)
film_structure       = Structure.from_file(film_POSCAR)
warn_if_not_slab_like(substrate_structure, 'substrate')
warn_if_not_slab_like(film_structure, 'film')

substrate_slab = wrap_as_slab(substrate_structure, substrate_miller)
film_slab      = wrap_as_slab(film_structure, film_miller)


## Configuration

In [ ]:
if not os.path.exists(general_folder):
    os.makedirs(general_folder)

vacuum_over_film = 20.0   # Å, vacuum left over the top of the film

# Search space for candidate generation -- one clear parameter per axis
angle_bounds          = (0.0, 180.0)  # twist angle of the film about the interface normal, degrees
n_twist_angles        = 6             # how many angles to sample in angle_bounds (1 = no twist search)
gap_bounds            = (1.5, 4.0)    # interlayer gap, Å
n_gaps                = 3             # how many gaps to sample in gap_bounds (1 = single fixed gap)
n_matches_per_angle   = 3             # ZSL matches kept per angle, ranked by lowest strain first
n_registries_per_match = 4            # symmetry-distinct in-plane registries kept per match

# Single overall cap: if angles × matches × registries × gaps exceeds this, the candidate
# list is subsampled evenly down to this many -- this is the one number to turn if you just
# want "give me roughly N heterostructures to look at".
max_total_candidates = 60

# ZSL matching tolerances -- the main lever for "different lattice and symmetry": for a
# substrate/film pair with very mismatched lattices or low symmetry you may need a larger
# max_interface_area or looser tolerances before any coherent match exists at all
max_interface_area  = 200.0  # Å², cap on the coincidence supercell area
max_area_ratio_tol  = 0.09
max_length_tol      = 0.03
max_angle_tol       = 0.03

# Whether to relax/rank the generated candidates with the MACE potential. Leave this off to
# just get the candidate POSCARs (e.g. to hand them to VASP/DFT instead) without touching
# them at all; every POSCAR is left untouched either way -- relaxation results go into a
# separate CONTCAR next to it, same as elsewhere in this repo.
run_mliap_relaxation = False
model_load_path       = 'large'

heterostructure_data = {
    'substrate_miller':      substrate_miller,
    'film_miller':           film_miller,
    'vacuum_over_film':      vacuum_over_film,
    'angle_bounds':          angle_bounds,
    'n_twist_angles':        n_twist_angles,
    'gap_bounds':            gap_bounds,
    'n_gaps':                n_gaps,
    'n_matches_per_angle':   n_matches_per_angle,
    'n_registries_per_match': n_registries_per_match,
    'max_total_candidates':  max_total_candidates,
    'max_interface_area':    max_interface_area,
    'max_area_ratio_tol':    max_area_ratio_tol,
    'max_length_tol':        max_length_tol,
    'max_angle_tol':         max_angle_tol,
    'run_mliap_relaxation':  run_mliap_relaxation,
}

with open(f'{general_folder}/heterostructure_data.json', 'w') as json_file:
    json.dump(heterostructure_data, json_file, indent=2)

# Copy the original slab POSCARs there, untouched, for provenance
os.system(f'cp {substrate_POSCAR} {general_folder}/POSCAR-substrate')
os.system(f'cp {film_POSCAR}      {general_folder}/POSCAR-film')


## Interface-building helpers

In [ ]:
zslgen = ZSLGenerator(
    max_area_ratio_tol=max_area_ratio_tol,
    max_area=max_interface_area,
    max_length_tol=max_length_tol,
    max_angle_tol=max_angle_tol,
    bidirectional=True,
)


def rotate_slab(slab, twist_angle):
    """Fresh copy of `slab`, rotated about the interface normal (z) by `twist_angle`
    degrees."""
    if abs(twist_angle) < 1e-9:
        return slab.copy()
    rotation = RotationTransformation(axis=[0, 0, 1], angle=twist_angle)
    return rotation.apply_transformation(slab.copy())


def match_strain(match):
    """Von Mises strain of a ZSL match -- lower means a less-distorted, more physically
    reasonable coincidence lattice."""
    return Deformation(match.match_transformation).green_lagrange_strain.von_mises_strain


def find_matches(rotated_film_slab, substrate_slab, n_keep):
    """All ZSL matches between the two slabs' in-plane lattices, sorted by ascending
    strain and capped to `n_keep`."""
    matches = list(zslgen(rotated_film_slab.lattice.matrix[:2], substrate_slab.lattice.matrix[:2], lowest=False))
    matches.sort(key=match_strain)
    return matches[:n_keep]


def build_supercell_slabs(rotated_film_slab, substrate_slab, match):
    """Build the film/substrate supercells implied by a ZSL match (same recipe
    CoherentInterfaceBuilder.get_interfaces uses internally, applied directly to our own
    pre-cut slabs instead of slabs it would otherwise cut itself from a bulk cell)."""
    film_transform = np.round(from_2d_to_3d(
        get_2d_transform(rotated_film_slab.lattice.matrix[:2], match.film_sl_vectors)
    )).astype(int)
    film_sl_slab = rotated_film_slab.copy()
    film_sl_slab.make_supercell(film_transform)

    sub_transform = np.round(from_2d_to_3d(
        get_2d_transform(substrate_slab.lattice.matrix[:2], match.substrate_sl_vectors)
    )).astype(int)
    sub_sl_slab = substrate_slab.copy()
    sub_sl_slab.make_supercell(sub_transform)

    return film_sl_slab, sub_sl_slab


def enumerate_registries(film_sl_slab, sub_sl_slab, n_keep):
    """Symmetry-distinct in-plane (lateral) registries for this film/substrate supercell
    pair, capped and evenly subsampled to `n_keep` if more are found."""
    reference_gap = float(np.mean(gap_bounds))
    reference_interface = Interface.from_slabs(
        substrate_slab=sub_sl_slab.copy(), film_slab=film_sl_slab.copy(),
        gap=reference_gap, vacuum_over_film=vacuum_over_film,
    )
    registries = [tuple(shift) for shift in reference_interface.get_shifts_based_on_adsorbate_sites()]
    if not registries:
        registries = [(0.0, 0.0)]
    if len(registries) > n_keep:
        idx = sorted(set(np.linspace(0, len(registries) - 1, n_keep).round().astype(int)))
        registries = [registries[i] for i in idx]
    return registries


def to_plain_structure(structure):
    """Interface/Slab objects carry extra site_properties that some ASE/pymatgen version
    combinations choke on when converting to ase.Atoms. Stripping to a plain Structure with
    just species/coords avoids that."""
    return Structure(structure.lattice, structure.species, structure.frac_coords, coords_are_cartesian=False)


def write_structure(structure, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    write_vasp(path, AseAtomsAdaptor.get_atoms(to_plain_structure(structure)), direct=True, sort=True)


## Adhesion energy (used only if `run_mliap_relaxation=True`)

\begin{equation}
    E_{\text{adhesion}} = \frac{E_{\text{interface}} - E_{\text{film}} - E_{\text{substrate}}}{S}
\end{equation}

analogous to `sul.get_surface_energy_of_formation`, but normalized by a single interface
(not the two free surfaces of an isolated slab). All three energies come from structures
sharing the same (strained) interface cell, and the result is converted to J/m² with the
same eV/Å² factor used elsewhere in this repo.


In [ ]:
def get_adhesion_energy_of_formation(interface_energy, film_energy, substrate_energy, interface_area):
    return (interface_energy - film_energy - substrate_energy) * 16.0218 / interface_area


## Generate the candidate interfaces

In [ ]:
angle_grid = np.linspace(*angle_bounds, n_twist_angles) if n_twist_angles > 1 else [angle_bounds[0]]
gap_grid   = np.linspace(*gap_bounds, n_gaps) if n_gaps > 1 else [float(np.mean(gap_bounds))]

candidates = []  # metadata only -- structures are (re)built once we know which ones survive the cap
for a_idx, angle in enumerate(angle_grid):
    rotated_film_slab = rotate_slab(film_slab, float(angle))
    matches = find_matches(rotated_film_slab, substrate_slab, n_matches_per_angle)
    if not matches:
        print(f'angle={angle:6.1f}°: no coherent match found')
        continue

    for m_idx, match in enumerate(matches):
        film_sl_slab, sub_sl_slab = build_supercell_slabs(rotated_film_slab, substrate_slab, match)
        registries = enumerate_registries(film_sl_slab, sub_sl_slab, n_registries_per_match)
        for r_idx, registry in enumerate(registries):
            for g_idx, gap in enumerate(gap_grid):
                candidates.append({
                    'angle_idx': a_idx, 'angle': float(angle),
                    'match_idx': m_idx, 'strain': float(match_strain(match)),
                    'registry_idx': r_idx, 'registry': registry,
                    'gap_idx': g_idx, 'gap': float(gap),
                    'film_sl_slab': film_sl_slab, 'sub_sl_slab': sub_sl_slab,
                })

print(f'{len(candidates)} candidate interface(s) before capping')

if len(candidates) > max_total_candidates:
    idx = sorted(set(np.linspace(0, len(candidates) - 1, max_total_candidates).round().astype(int)))
    candidates = [candidates[i] for i in idx]
    print(f'subsampled down to max_total_candidates={max_total_candidates}')

print(f'{len(candidates)} candidate interface(s) to write')


In [ ]:
for c_idx, candidate in enumerate(candidates):
    candidate_folder = (
        f'{general_folder}/angle_{candidate["angle_idx"]}/match_{candidate["match_idx"]}/'
        f'registry_{candidate["registry_idx"]}/gap_{candidate["gap_idx"]}'
    )
    reference_folder = f'{general_folder}/angle_{candidate["angle_idx"]}/match_{candidate["match_idx"]}/reference'

    film_sl_slab = candidate['film_sl_slab']
    sub_sl_slab  = candidate['sub_sl_slab']

    interface = Interface.from_slabs(
        substrate_slab=sub_sl_slab.copy(), film_slab=film_sl_slab.copy(),
        in_plane_offset=candidate['registry'], gap=candidate['gap'], vacuum_over_film=vacuum_over_film,
    )
    interface_area = np.linalg.norm(np.cross(interface.lattice.matrix[0], interface.lattice.matrix[1]))

    write_structure(interface, f'{candidate_folder}/POSCAR')
    # Isolated film/substrate references only depend on (angle, match), not on registry/gap,
    # so they're written once per (angle, match) and shared -- sul.read_energy will only
    # actually relax them the first time it sees this folder.
    write_structure(film_sl_slab, f'{reference_folder}/film/POSCAR')
    write_structure(sub_sl_slab,  f'{reference_folder}/substrate/POSCAR')

    with open(f'{candidate_folder}/candidate_data.json', 'w') as json_file:
        json.dump({
            'substrate_miller': substrate_miller,
            'film_miller':       film_miller,
            'twist_angle':       candidate['angle'],
            'strain':            candidate['strain'],
            'registry':          candidate['registry'],
            'gap':               candidate['gap'],
            'interface_area':    interface_area,
            'reference_folder':  reference_folder,
        }, json_file, indent=2)

    candidates[c_idx]['candidate_folder'] = candidate_folder
    candidates[c_idx]['reference_folder'] = reference_folder
    candidates[c_idx]['interface_area']   = interface_area

print(f'Wrote {len(candidates)} candidate interface POSCARs under {general_folder}/')


## Optional: relax and rank with the MACE potential

Only runs if `run_mliap_relaxation = True` above. Every candidate `POSCAR` written above is
left as-is; this reads/writes `CONTCAR` and energy files alongside it via the existing
`sul.read_energy`/`libraries.model` pipeline, exactly like `iterate-slabs.ipynb` and
`H-absorption.ipynb` do.


In [ ]:
results = []
if run_mliap_relaxation:
    for candidate in candidates:
        film_energy      = sul.read_energy(f'{candidate["reference_folder"]}/film',      model_load_path=model_load_path)
        substrate_energy = sul.read_energy(f'{candidate["reference_folder"]}/substrate', model_load_path=model_load_path)
        interface_energy = sul.read_energy(candidate['candidate_folder'],                  model_load_path=model_load_path)

        if any(e is None or np.isnan(e) for e in (film_energy, substrate_energy, interface_energy)):
            print(f'{candidate["candidate_folder"]}: relaxation/energy failed, skipping')
            continue

        adhesion_energy = get_adhesion_energy_of_formation(
            interface_energy, film_energy, substrate_energy, candidate['interface_area']
        )

        with open(f'{candidate["candidate_folder"]}/candidate_data.json') as json_file:
            candidate_data = json.load(json_file)
        candidate_data.update({
            'film_energy':       film_energy,
            'substrate_energy':  substrate_energy,
            'interface_energy':  interface_energy,
            'adhesion_energy':   adhesion_energy,
        })
        with open(f'{candidate["candidate_folder"]}/candidate_data.json', 'w') as json_file:
            json.dump(candidate_data, json_file, indent=2)

        results.append({'folder': candidate['candidate_folder'], 'adhesion_energy': adhesion_energy})
        print(f'{candidate["candidate_folder"]}: E_adhesion = {adhesion_energy:.4g} J/m²')

    results.sort(key=lambda r: r['adhesion_energy'])
    with open(f'{general_folder}/heterostructure_results.json', 'w') as json_file:
        json.dump(results, json_file, indent=2)

    if results:
        print(f'\nBest: {results[0]["folder"]}  E_adhesion = {results[0]["adhesion_energy"]:.4g} J/m²')
else:
    print('run_mliap_relaxation is False -- candidate POSCARs were generated but not relaxed/ranked.')


## Ranking plot (only if `run_mliap_relaxation = True` and candidates were scored)

In [ ]:
if results:
    labels   = [r['folder'].replace(f'{general_folder}/', '') for r in results]
    energies = [r['adhesion_energy'] for r in results]

    plt.figure(figsize=(max(4, len(labels) * 0.5), 4))
    plt.plot(energies, 'o-')
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.ylabel(r'$E_{\text{adhesion}}$ (J/m$^2$)')
    plt.axhline(0, color='grey', linewidth=0.8)
    plt.tight_layout()
    plt.savefig(f'{general_folder}/adhesion_ranking.png', dpi=200)
    plt.show()
